# Assignment 1: Data Preprocessing

## 1. Introduction

Silicon wafer manufacturing is a complex and highly sensitive process where even minor deviations can cause defects, reducing yield and increasing costs. Wafer defects often cluster into specific spatial patterns on the wafer surface, which are highly indicative of particular equipment failures or process flaws.

The importance of automatic defect classification cannot be overstated; manual inspection is slow, expensive, and prone to human error. Machine learning plays a pivotal role here by automatically recognizing these spatial defect patterns, enabling rapid root-cause analysis and timely corrective actions in the manufacturing process.

The objective of the selected research paper is to develop an automated machine learning approach to detect and classify these defect patterns accurately. The objective of this assignment is to lay the foundation for reproducing that work by thoroughly inspecting and preprocessing the WM-811K dataset, ensuring the data is clean, formatted, and ready for model training.

## 2. Research Paper Information

- **Paper title**: [Placeholder: Paper Title]
- **Authors**: [Placeholder: Authors]
- **Journal/Conference**: [Placeholder: Journal/Conference]
- **Year**: [Placeholder: Year]
- **DOI**: [Placeholder: DOI]
- **URL**: [Placeholder: URL]

## 3. Dataset Description

- **Dataset name**: WM-811K / Silicon Wafer Defect Dataset
- **Dataset source**: Extracted from local archive (`archive.zip`)
- **Dataset file structure**: The primary dataset file is `LSWMD.pkl`, stored under `data/raw/`.
- **Wafer map representation**: Wafer maps are represented as 2D arrays where each element corresponds to a die (e.g., 0 = background, 1 = normal, 2 = defect).
- **Target/label information**: `failureType` column indicates the defect pattern classification.
- **Relevant attributes**: `waferMap`, `dieSize`, `lotName`, `waferIndex`, `trianTestLabel`, `failureType`.

In [ ]:
import sys
from pathlib import Path

# Ensure src directory is in the Python path for Colab execution
sys.path.append(str(Path.cwd().parent))

from src.data_loader import load_raw_data

df_raw = load_raw_data()

print("Dataset shape:", df_raw.shape)
print("\nColumn names:", df_raw.columns.tolist())
print("\nData types:\n", df_raw.dtypes)
display(df_raw.head())

## 4. Initial Dataset Inspection

In [ ]:
num_samples = len(df_raw)
num_attributes = len(df_raw.columns)
missing_values = df_raw.isnull().sum()
duplicate_records = df_raw.duplicated().sum()
unique_labels = df_raw['failureType'].astype(str).unique() if 'failureType' in df_raw.columns else []

print(f"Number of samples: {num_samples}")
print(f"Number of attributes: {num_attributes}")
print(f"Duplicate records: {duplicate_records}")
print(f"Unique labels/classes: {unique_labels}")
display(missing_values)

## 5. Attribute-Level Preprocessing Analysis

| Attribute | Original Representation | Problem | Technique | Reason | Final Representation |
| :--- | :--- | :--- | :--- | :--- | :--- |
| `waferMap` | 2D Array / List of Lists | Variable dimensions | Resizing / Padding | CNNs require fixed input dimensions | Fixed-size 2D Matrix (e.g., NumPy array) |
| `failureType` | Mixed/String | Non-numeric, contains nulls | Encoding, Filtering | ML models require numeric targets | Integer encoded labels |
| `dieSize` | Float/Int | Sometimes missing | Extraction from waferMap | Needed for feature consistency | Integer |
| `lotName` | String | Irrelevant for ML | Drop / Ignore | Does not contribute to defect classification | Dropped |
| `waferIndex` | Integer | Irrelevant for ML | Drop / Ignore | Just an identifier | Dropped |
| `trianTestLabel` | List of List/String | Array format | Flattening/Mapping | Difficult to filter directly | String / Boolean mask |

## 6. Missing Value Analysis

In [ ]:
missing_count = df_raw.isnull().sum()
missing_percentage = (missing_count / len(df_raw)) * 100

import pandas as pd
missing_df = pd.DataFrame({'Missing Count': missing_count, 'Percentage (%)': missing_percentage})
display(missing_df[missing_df['Missing Count'] > 0])

## 7. Duplicate Analysis

In [ ]:
duplicate_count = df_raw.duplicated().sum()
duplicate_pct = (duplicate_count / len(df_raw)) * 100

print(f"Duplicates: {duplicate_count} ({duplicate_pct:.2f}%)")

## 8. Invalid Data Analysis

We inspect for unusable wafer maps (e.g., dimensions of 0 or invalid entries) and records without proper labels.

In [ ]:
# Code to check for invalid representations
pass

## 9. Label Processing

In [ ]:
if 'failureType' in df_raw.columns:
    # Note: failureType in WM-811K is often embedded as [['Pattern']]
    df_raw['label_flat'] = df_raw['failureType'].apply(lambda x: x[0][0] if isinstance(x, np.ndarray) and len(x)>0 else x)
    
    class_counts = df_raw['label_flat'].value_counts()
    print("Class Counts:\n", class_counts)
    
    # Implement mapping dictionary for later use
    label_mapping = {label: idx for idx, label in enumerate(class_counts.index)}
    print("\nMapping:\n", label_mapping)

## 10. Wafer Map Representation

We must standardize the 2D arrays so they have compatible formats for future downstream tasks.

In [ ]:
# Demonstrating standardization
pass

## 11. Final Preprocessing Pipeline

Executing the full pipeline using `src.preprocessing`.

In [ ]:
from src.preprocessing import run_preprocessing_pipeline

df_processed = run_preprocessing_pipeline(df_raw)
print("Processed dataset shape:", df_processed.shape)

## 12. Save Processed Dataset

In [ ]:
from src.config import PROCESSED_DATA_DIR

output_path = PROCESSED_DATA_DIR / "processed_data.pkl"
df_processed.to_pickle(output_path)
print(f"Saved to {output_path}")

## 13. Output Tables/Figures

Saving the descriptive summaries.

In [ ]:
from src.config import TABLES_DIR, FIGURES_DIR
import matplotlib.pyplot as plt

# e.g., class_counts.to_csv(TABLES_DIR / 'class_distribution.csv')
pass

## 14. Conclusion

In this assignment, the raw WM-811K dataset was loaded and rigorously inspected. Preprocessing focused on resolving varying missing data elements, properly flattening complex label representations into a 1D numerical mapping, and establishing a consistent structure for the `waferMap` 2D arrays. 

These techniques were essential because the original data structure contained nested arrays and categorical strings unsuitable for direct mathematical computation. The resulting standardized, purely numeric dataset is now reliably prepared for exploratory data analysis and future machine learning assignments without compromising the integrity of the original labels.